<div align="center">

# Rerank fse candidates with a local cross-encoder

Let fse find 100 candidates, then let a cross-encoder of your choice put the best ones on top.

</div>

## When do you need this?

A static model gives every text one vector, and the search compares vectors. The question and the document never meet.
A **cross-encoder** reads the question and one document together and returns a score for the pair. It often ranks better, but not on every kind of data, and it computes one transformer pass per pair, so it can read only a few candidates.
The usual pattern has two stages: a fast first stage finds candidates, the cross-encoder orders them again.

**fse stores vectors, not texts.** It has no reranker and calls no service. You fetch the candidate texts from your own source (a dict, a database, files) and choose any local cross-encoder.

## The story in a table

| Step | What happens | Call |
|---|---|---|
| 1 | Write the documents as vectors with their IDs | `ShardSink(doc_ids=True)`, `encode` |
| 2 | Find 100 candidates per question | `VectorStore.search` |
| 3 | Fetch the candidate texts from your source | your dict or database |
| 4 | Score each (question, candidate) pair | `CrossEncoder.predict` |
| 5 | Optional: fuse with BM25 before reranking | Reciprocal Rank Fusion |

## 1. Documents, questions and judgments

We use [NanoSciFact](https://huggingface.co/datasets/mteb/NanoSciFactRetrieval), one of the 13 NanoBEIR tasks: 2,919 scientific abstracts and 50 claims, each with the abstracts that support or refute it.
The judgments let us measure **NDCG@10**: 1.0 means the relevant documents fill the top of the list, 0 means none is in the top 10.
The first stage is [potion-retrieval-32M](https://huggingface.co/minishlab/potion-retrieval-32M), a static model trained for retrieval.
The [shared setup](README.md#run-a-recipe) installs the pinned packages. This recipe also needs `sentence-transformers`, `bm25s` and `PyStemmer` from `requirements.txt`.

In [1]:
from _helpers import Recipe
import _rerank
import numpy as np
import torch
from sentence_transformers import CrossEncoder

import fse

recipe = Recipe()
work = recipe.work
docs, questions, judged = _rerank.nano_task(recipe)
print("Documents:", len(docs), "Questions:", len(questions))

POTION = ("minishlab/potion-retrieval-32M", "6fc8051fab2a1e0ee76689cf08c853792ac285e7")
encoder = fse.StaticModel.from_pretrained(_rerank.snapshot(recipe, *POTION), normalize=True).encoder

Recipe run: 2026-11-08. Generated files are temporary.
Apple M1 Pro. macOS-27.0.1-arm64-arm-64bit-Mach-O. Python 3.13.16.
Documents: 2919 Questions: 50


## 2. Write a store with document IDs

The store keeps only vectors and an int64 ID per row. The texts stay in `docs`, our stand-in for your own source.

In [2]:
doc_ids = list(docs)
assert all(str(int(d)) == d for d in doc_ids)  # these IDs are integers written as text
with fse.ShardSink(work / "store", encoder, doc_ids=True) as sink:
    encoder.encode([docs[d] for d in doc_ids], doc_ids=[int(d) for d in doc_ids], sink=sink)
store = fse.VectorStore.open(work / "store")
print("Rows:", len(store))

Rows: 2919


## 3. Find 100 candidates per question

`search(mode="exact")` compares every row. `found.doc_ids` gives the IDs that lead back to the texts.

In [3]:
qids = list(questions)
found = store.search(encoder.encode([questions[q] for q in qids]), 100, mode="exact")
candidates = {q: [str(d) for d in found.doc_ids[i]] for i, q in enumerate(qids)}
print("fse alone, NDCG@10:", round(_rerank.mean_ndcg(candidates, judged), 3))

fse alone, NDCG@10: 0.683


## 4. Choose a reranker

Any [Sentence Transformers cross-encoder](https://sbert.net/docs/cross_encoder/usage/usage.html) with one relevance score per pair works. Pin its revision.
We start with [ms-marco-MiniLM-L6-v2](https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2), small (about 90 MB) and widely used.
For a larger one, replace the two lines with [Qwen3-Reranker-0.6B](https://huggingface.co/Qwen/Qwen3-Reranker-0.6B) at revision `e61197ed45024b0ed8a2d74b80b4d909f1255473` (about 1.2 GB, and `model_kwargs={"dtype": torch.float16}` on a GPU). It brings its own instruction prompt.
The reranker may use your GPU (CUDA or Apple MPS). It is not part of fse.

In [4]:
RERANKER = ("cross-encoder/ms-marco-MiniLM-L6-v2", "233902d25c440f23af6f7d6e94d2946bac0bee0a")
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
reranker = CrossEncoder(_rerank.snapshot(recipe, *RERANKER), device=device, max_length=512)


def rerank(question, ranked, top):
    """The first `top` candidates ordered by the reranker, then the rest unchanged."""
    head = ranked[:top]
    scores = np.asarray(reranker.predict([(question, docs[d]) for d in head], batch_size=32))
    if scores.shape != (len(head),) or not np.isfinite(scores).all():
        raise ValueError("the reranker must return one finite score per pair")
    order = np.argsort(-scores, kind="stable")  # ties keep the first-stage order
    return [head[i] for i in order] + ranked[top:]


for top in (10, 100):
    reranked = {q: rerank(questions[q], candidates[q], top) for q in qids}
    print(f"fse + reranker on the top {top:3d}, NDCG@10:", round(_rerank.mean_ndcg(reranked, judged), 3))

fse + reranker on the top  10, NDCG@10: 0.739


fse + reranker on the top 100, NDCG@10: 0.737


Reranking the top 10 only reorders what fse already found. Reranking the top 100 can lift a relevant document from rank 60 into the top 10, but it costs ten times the pairs.
Whether it helps depends on the reranker and the domain: ms-marco-MiniLM-L6-v2 learned from web search questions, and scientific claims are a different kind of question.

## 5. Optional: fuse with BM25 first

BM25 ranks by shared words, so it finds exact terms that a static model may blur. **Reciprocal Rank Fusion** (RRF) adds `1 / (60 + rank)` over both lists ([Cormack, Clarke and Büttcher, SIGIR 2009](https://doi.org/10.1145/1571941.1572114)). It needs no scores and no training.
[bm25s](https://github.com/xhluca/bm25s) uses the Lucene variant with k1 = 1.5 and b = 0.75.

In [5]:
import bm25s
import Stemmer

stemmer = Stemmer.Stemmer("english")
bm25 = bm25s.BM25()
bm25.index(bm25s.tokenize([docs[d] for d in doc_ids], stopwords="en", stemmer=stemmer, show_progress=False), show_progress=False)
tokens = bm25s.tokenize([questions[q] for q in qids], stopwords="en", stemmer=stemmer, show_progress=False)
hits, _ = bm25.retrieve(tokens, k=100, show_progress=False)
lexical = {q: [doc_ids[j] for j in hits[i]] for i, q in enumerate(qids)}
fused = {q: _rerank.rrf(candidates[q], lexical[q])[:100] for q in qids}
print("BM25 alone, NDCG@10:      ", round(_rerank.mean_ndcg(lexical, judged), 3))
print("RRF(fse, BM25), NDCG@10:  ", round(_rerank.mean_ndcg(fused, judged), 3))
fused_reranked = {q: rerank(questions[q], fused[q], 10) for q in qids}
print("RRF + reranker on the top 10:", round(_rerank.mean_ndcg(fused_reranked, judged), 3))

BM25 alone, NDCG@10:       0.71
RRF(fse, BM25), NDCG@10:   0.749


RRF + reranker on the top 10: 0.752


## Result

fse found the candidates, the texts came from our own dict, and a local cross-encoder ordered them again. The numbers above are for one task with 50 questions. They vary between tasks and rerankers, so measure on your own data before choosing.
The repository's reranker study measures such arms on all 13 NanoBEIR tasks (see the [measurement guide](../docs/benchmarks/measurement.md)). It fuses lists of 1,000 documents and breaks ties differently, so its RRF values can differ from this recipe's.
This recipe measures no time on purpose: the cost of a reranker depends on its size, the text length, the number of candidates and your hardware.

## Before using your own data

- **Texts.** The store holds no texts. Keep a lookup from document ID to text that is as fast as your search, or the lookup becomes the slow part.
- **Length.** `max_length=512` cuts each pair to 512 tokens. Split long documents into chunks with the same document ID, search with `max_per_doc=1`, and give the reranker the text of the chunk that was found (`found.ids` is its row), not the whole document.
- **Cost.** A reranker computes one pass per pair: 100 candidates for 1,000 questions are 100,000 passes. Choose the depth by measuring both quality and cost.
- **Licenses.** The model and dataset terms apply separately from fse's license.

## What next?

- [Document IDs](../docs/document-ids.md)
- [Vector stores in detail](../docs/architecture/store.md)
- [External indexes](10-external-ann-indexes.ipynb) for stores too large for an exact search

Back to the [recipe index](README.md) or the [main README](../README.md).

## Clean up

The helper keeps generated files in a temporary folder. Run this after you finish exploring the results.

In [6]:
store.close()
recipe.close()